In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.inspection import DecisionBoundaryDisplay
import time

In [ ]:
col = input("select number of color (from 1 to 16):")

In [ ]:
HSV = True
onlyHS = True

photo = cv2.imread("S1_train/7164_10.png") 
orig = photo.copy()
print("Photo shape: ", photo.shape)

if HSV:
    photo = cv2.cvtColor(photo, cv2.COLOR_BGR2HSV)

gt_file = ("S1_train/7164_"+col+"_GT_10.png")
gt = cv2.imread(gt_file)
print(gt_file)
gt = cv2.cvtColor(gt, cv2.COLOR_BGR2HSV)


In [ ]:

t_inici = time.time()

# get non-zero values in the mask
mask = np.any(gt != [0, 0, 0], axis=-1) 

# get indices of GT pixels
ind = np.where(mask)

# new array containing only GT colors
gt_colors = photo[ind[0], ind[1], :]
print("Positive examples: ", gt_colors.shape)

if onlyHS:
    gt_colors = gt_colors[:, 0:2]

# stats of GT colors
print("Mean of positive examples: ", np.mean(gt_colors, axis=0))
print("Std of positive examples: ", np.std(gt_colors, axis=0))

# Now do the same but for the rest of colors
ind = np.where(~mask)
# new array containing only rest of colors
rest_colors = photo[ind[0], ind[1], :]
if onlyHS:
    rest_colors = rest_colors[:, 0:2]

print("Negative examples: ",rest_colors.shape)

# stats of other colors
print("Mean of negative examples: ", np.mean(rest_colors,axis=0))
print("Std of negative examples: ", np.std(rest_colors, axis=0))

# positive + negative examples
X = np.concatenate( (gt_colors, rest_colors))
y = np.concatenate( (np.ones(gt_colors.shape[0]), np.zeros(rest_colors.shape[0]))) # predict "1" for positives, "0" for negatives
datasets = [(X,y)]

t_dataset = time.time()
print ("time dataset: ", t_dataset-t_inici)

In [ ]:
# Based on "Classifier comparison" de Gaël Varoquaux, Andreas Müller; Modified for documentation by Jaques Grobler
# License: BSD 3 clause

t_inici = time.time()

names = [
    "NearestNeighbors",
    "RBF_SVM",
    "DecisionTree",
    "NaiveBayes",
]

classifiers = [
    KNeighborsClassifier(14),
    SVC(gamma=10, C=1),
    DecisionTreeClassifier(max_depth=5),    #sembla que depth=5 és la òptima
    GaussianNB(),
]

scaler = StandardScaler()

figure = plt.figure(figsize=(4*len(names), 4*len(datasets)))
i = 1
# iterate over datasets
for ds_cnt, ds in enumerate(datasets):
    # preprocess dataset, split into training and test part
    X, y = ds
    X = scaler.fit_transform(X) # required by SVM and others
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    # just plot the dataset first
    cm = plt.cm.RdBu
    cm_bright = ListedColormap(["#FF0000", "#0000FF"])
    ax = plt.subplot(len(datasets), len(classifiers) + 1, i)
    if ds_cnt == 0:
        ax.set_title("Input data")
    # Plot the training points
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors="k")
    # Plot the testing points
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, alpha=0.6, edgecolors="k")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(())
    ax.set_yticks(())
    i += 1

    t_dataset = time.time()
    print ("time dataset: ", t_dataset-t_inici)
    
    # iterate over classifiers
    for name, clf in zip(names, classifiers):
        ax = plt.subplot(len(datasets), len(classifiers) + 1, i)
        print("Training ", name, " with parameters ", clf)
        clf.fit(X_train, y_train)
        score = clf.score(X_test, y_test)
        
        y_pred = clf.predict(X_test)
        f1 = f1_score(y_test, y_pred)
        
        if onlyHS:
            DecisionBoundaryDisplay.from_estimator(clf, X, cmap=cm, alpha=0.8, ax=ax, eps=0.5)

        # Plot the training points
        ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors="k")
        # Plot the testing points
        ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, edgecolors="k", alpha=0.6)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xticks(())
        ax.set_yticks(())
        if ds_cnt == 0: ax.set_title(name)
        ax.text(
            x_max - 0.3,
            y_min + 0.3,
            ("%.3f" % score).lstrip("0") +  "/" + ("%.3f" % f1).lstrip("0"),
            size=15,
            horizontalalignment="right",
        )
        t_training = time.time()
        print ("time training", name, ": ", t_training-t_dataset)
        i += 1

plt.tight_layout()
plt.show()

In [ ]:
# Make predictions

#Indicate the image to be segmented
file_pred = "S2_test/papers_colors_v3_small_scene4_0371_25.png"
paper = col
photo = cv2.imread(file_pred)
orig = photo.copy()

photo = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)


if HSV:
    photo = cv2.cvtColor(photo, cv2.COLOR_RGB2HSV)

figure = plt.figure(figsize=(4*len(names), 4*len(datasets)))
# iterate over datasets

t_inici = time.time()

for ds_cnt, ds in enumerate(datasets):
    # iterate over classifiers
    for name, clf in zip(names, classifiers):
        print(name)
        # predict on the same image
        photo_flat = np.reshape(photo, (photo.shape[0]*photo.shape[1], 3))
        if onlyHS:
            photo_flat = photo_flat[:, 0:2]

        photo_flat = scaler.transform(photo_flat)
        photo_segmented = clf.predict(photo_flat)
        
        photo_segmented = np.reshape(photo_segmented, (photo.shape[0], photo.shape[1]))
        B = (orig[:,:,0] * photo_segmented)/255.
        G = (orig[:,:,1] * photo_segmented)/255.
        R = (orig[:,:,2] * photo_segmented)/255.

        filename = "results/"+file_pred[:-4]+"_"+name+"_"+paper+"_pred.png"
        
        out = cv2.merge((R,G,B))
        out_save = cv2.merge((B,G,R))*255
        cv2.imwrite(filename, out_save)
        plt.figure()
        plt.imshow(out)
        t_final = time.time()
        print ("time detection", name, ": ", t_final-t_inici)

In [ ]:
# Precision, Recall, F1 - S2

import pandas as pd
import glob

folder = "S2_test"
GT_files = glob.glob(folder+"/*GT*.png")
GT_files.sort()
results_files = glob.glob(folder+"/*pred.png")
results_files.sort()

df = pd.DataFrame()
idx = 0

for gt in GT_files:
        img_GT = cv2.imread(gt)
        img_GT = img_GT
        for res in results_files:
            if gt[-17:-14] in res and gt[-20] == res[-10]:
                result = cv2.imread(res)
                result = result
                FN_img = cv2.subtract(img_GT, result)
                FP_img = cv2.subtract(result, img_GT)
                TP_img = cv2.bitwise_and(result, img_GT)
                #TP_img = cv2.cvtColor(TP_img, cv2.COLOR_RGB2BGR)

                FN_img_g = cv2.cvtColor(FN_img, cv2.COLOR_BGR2GRAY)
                FP_img_g = cv2.cvtColor(FP_img, cv2.COLOR_BGR2GRAY)
                TP_img_g = cv2.cvtColor(TP_img, cv2.COLOR_BGR2GRAY)
                
                size = FN_img.shape
                total = size[0]*size[1]

                FN_im = FN_img_g.flatten()
                FP_im = FP_img_g.flatten()
                TP_im = TP_img_g.flatten()

                FN_list = FN_im[FN_im>0]
                FP_list = FP_im[FP_im>0]
                TP_list = TP_im[TP_im>0]

                FN = len(FN_list)
                FP = len(FP_list)
                TP = len(TP_list)

                print ("Total: ", total)
                print ("FN: ", FN)
                print ("FP: ", FP)
                print ("TP: ", TP)
                
                TN = total - TP - FP - FN

                print ("TN: ", TN)
                
                if TP == 0: 
                    prec = 0
                    recall = 0
                    F1 = 0
                elif TP != 0:
                    #print (res)
                    prec = TP / (TP+FP)
                    #print (prec)
                    recall = TP / (TP+FN)
                    #print (recall)
                    F1 = 2*(prec*recall)/(prec + recall)
                
                
                if "DecisionTree" in res: clas = "Decision Tree"
                elif "NaiveBayes" in res: clas = "Naive Bayes"
                elif "NearestNeighbors" in res: clas = "Nearest Neighbours"
                elif "RandomForest" in res: clas = "Random Forest" 
                elif "RBF_SVM" in res: clas = "RBF SVM" 

                
                if "p1" in gt:
                    paper = 1
                elif "p2" in gt:
                    paper = 2
                elif "p3" in gt:
                    paper = 3
                elif "p4" in gt:
                    paper = 4
                
                df.at[idx, "File"] = res
                df.at[idx, "Image"] = gt[-23]
                df.at[idx, "Paper"] = paper
                df.at[idx, "Classifier"] = clas
                df.at[idx, "TP"] = TP
                df.at[idx, "FP"] = FP
                df.at[idx, "FN"] = FN
                df.at[idx, "Precision"] = prec
                df.at[idx, "Recall"] = recall
                df.at[idx, "F1"] = F1
                
                idx += 1

#print(df)
df.to_excel("results/S2_Classifiers_F1.xlsx") 